# 02 – Vérification des données nettoyées

**Objectif** : valider que `src/preprocessing.py` a bien fait son travail.
On refait les contrôles du notebook 01 et on vérifie que les problèmes repérés ont disparu.

> Prérequis : `python -m src.preprocessing` (génère `data/processed/matches_clean.csv`).

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data_ingestion import load_raw
from src.preprocessing import load_clean

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

## 1. Avant / après

In [ ]:
raw = load_raw()
clean = load_clean()

print(f"Brut    : {raw.shape[0]} lignes, {raw.shape[1]} colonnes")
print(f"Nettoyé : {clean.shape[0]} lignes, {clean.shape[1]} colonnes")
print(f"Lignes supprimées : {raw.shape[0] - clean.shape[0]}")

In [ ]:
clean.head()

## 2. Types et valeurs manquantes

In [ ]:
clean.info()

In [ ]:
missing = (clean.isna().mean() * 100).round(2)
missing[missing > 0]

Les colonnes essentielles (`Date`, équipes, buts, `FTR`) ne doivent avoir **aucune** valeur manquante.
Les stats de tirs (`HS`, `AS`, `HST`, `AST`) peuvent en avoir quelques-unes : on verra comment les gérer à l'étape 3.

## 3. Statistiques descriptives

In [ ]:
num_cols = ["FTHG", "FTAG"] + [c for c in config.STAT_COLS if c in clean.columns]
clean[num_cols].describe().round(2)

**À vérifier** : minimum ≥ 0 partout, maximum plausible (pas de score ou de nombre de tirs absurde),
moyenne et médiane proches (sinon distribution très asymétrique), `count` = nombre de lignes pour les buts.

## 4. Cohérence temporelle

In [ ]:
print("Dates triées :", clean["Date"].is_monotonic_increasing)
print("Période      :", clean["Date"].min().date(), "→", clean["Date"].max().date())
clean.groupby("Season").agg(nb_matchs=("Date", "size"), debut=("Date", "min"), fin=("Date", "max"))

## 5. La cible

In [ ]:
print(clean[config.TARGET].value_counts())
print(f"\nTaux de victoire à domicile : {clean[config.TARGET].mean():.1%}")

baseline = 1 - clean[config.TARGET].mean()
print(f"Baseline (toujours prédire 'pas de victoire à domicile') : {baseline:.1%}")

## 6. Contrôles finaux

In [ ]:
print("Doublons                 :", clean.duplicated(subset=["Date", "HomeTeam", "AwayTeam"]).sum())
print("Noms d'équipes uniques   :", len(set(clean["HomeTeam"]) | set(clean["AwayTeam"])))
print("Équipe contre elle-même  :", (clean["HomeTeam"] == clean["AwayTeam"]).sum())
sorted(set(clean["HomeTeam"]) | set(clean["AwayTeam"]))

Relis la liste des équipes : un même club ne doit pas apparaître sous deux noms différents
(faute de frappe, abréviation...). Si c'est le cas, il faudra l'harmoniser dans `preprocessing.py`.

➡️ **Étape 3** : feature engineering (`src/features.py`).